In [1]:
# ======================================================
# XGBoost Model Training - Import Required Libraries
# ======================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Path Handling
from pathlib import Path

# Train-Test Split
from sklearn.model_selection import train_test_split

# Handle Imbalanced Dataset
from imblearn.over_sampling import SMOTE

# XGBoost Classifier
from xgboost import XGBClassifier

# Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# Cross Validation
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

# Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Save Model
import joblib

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [3]:
# ======================================================
# Section 2: Load Feature Engineered Dataset
# ======================================================

# Define dataset path
data_path = Path("../data/processed/application_train_feature_engineered.csv")

# Load dataset
feature_df = pd.read_csv(data_path)



In [4]:
feature_df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CREDIT_PER_PERSON,YEARS_EMPLOYED,EMPLOYMENT_RATIO,AGE,AGE_GROUP,INCOME_GROUP,LOAN_BURDEN,EXT_SOURCE_MEAN,TOTAL_DOCUMENTS,TOTAL_CONTACT_FLAGS
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,406597.50,1.745205,0.067329,25.920548,20-30,High,0.060749,0.161787,1,4
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,646751.25,3.254795,0.070862,45.931507,40-50,Very High,0.027598,0.466757,1,4
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,135000.00,0.616438,0.011814,52.180822,50-60,Very Low,0.050000,0.642739,0,5
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,156341.25,8.326027,0.159905,52.068493,50-60,Low,0.094941,0.650442,1,3
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,513000.00,8.323288,0.152418,54.608219,50-60,Low,0.042623,0.322738,1,3


In [5]:
feature_df.isnull().sum().sum()

np.int64(9152665)

In [6]:
# Total missing values
print("Total Missing Values:", feature_df.isnull().sum().sum())

# Missing values per column
missing = feature_df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

missing.head(20)

Total Missing Values: 9152665


COMMONAREA_MODE             214865
COMMONAREA_MEDI             214865
COMMONAREA_AVG              214865
NONLIVINGAPARTMENTS_MODE    213514
NONLIVINGAPARTMENTS_AVG     213514
NONLIVINGAPARTMENTS_MEDI    213514
FONDKAPREMONT_MODE          210295
LIVINGAPARTMENTS_AVG        210199
LIVINGAPARTMENTS_MEDI       210199
LIVINGAPARTMENTS_MODE       210199
FLOORSMIN_MODE              208642
FLOORSMIN_MEDI              208642
FLOORSMIN_AVG               208642
YEARS_BUILD_AVG             204488
YEARS_BUILD_MEDI            204488
YEARS_BUILD_MODE            204488
OWN_CAR_AGE                 202929
LANDAREA_MODE               182590
LANDAREA_AVG                182590
LANDAREA_MEDI               182590
dtype: int64

In [7]:
feature_df.replace([np.inf, -np.inf], np.nan, inplace=True)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CREDIT_PER_PERSON,YEARS_EMPLOYED,EMPLOYMENT_RATIO,AGE,AGE_GROUP,INCOME_GROUP,LOAN_BURDEN,EXT_SOURCE_MEAN,TOTAL_DOCUMENTS,TOTAL_CONTACT_FLAGS
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,406597.50,1.745205,0.067329,25.920548,20-30,High,0.060749,0.161787,1,4
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,646751.25,3.254795,0.070862,45.931507,40-50,Very High,0.027598,0.466757,1,4
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,135000.00,0.616438,0.011814,52.180822,50-60,Very Low,0.050000,0.642739,0,5
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,156341.25,8.326027,0.159905,52.068493,50-60,Low,0.094941,0.650442,1,3
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,513000.00,8.323288,0.152418,54.608219,50-60,Low,0.042623,0.322738,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,254700.00,0.646575,0.025303,25.553425,20-30,Medium,0.108198,0.413601,1,3
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,269550.00,1000.665753,17.580890,56.917808,50-60,Very Low,0.044524,0.115992,1,3
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,677664.00,21.701370,0.529266,41.002740,40-50,Medium,0.044239,0.499536,1,4
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,185053.50,13.112329,0.400134,32.769863,30-40,High,0.054592,0.587593,1,3


In [8]:
# Select numerical columns
numeric_cols = feature_df.select_dtypes(include=["number"]).columns

# Exclude target column
numeric_cols = numeric_cols.drop("TARGET")

# Fill missing values with median
feature_df[numeric_cols] = feature_df[numeric_cols].fillna(
    feature_df[numeric_cols].median()
)

In [9]:
categorical_cols = feature_df.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    feature_df[col] = feature_df[col].fillna(feature_df[col].mode()[0])

C:\Users\ASUS\AppData\Local\Temp\ipykernel_460\1945393641.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = feature_df.select_dtypes(include=["object", "category"]).columns


In [10]:
print("Remaining Missing Values:", feature_df.isnull().sum().sum())

Remaining Missing Values: 0


In [13]:
# ======================================================
# Section 4: Separate Features and Target
# ======================================================

# Features (Independent Variables)
X = feature_df.drop(columns=["TARGET"])

# Target (Dependent Variable)
y = feature_df["TARGET"]

print("Feature Matrix Shape :", X.shape)
print("Target Vector Shape  :", y.shape)

Feature Matrix Shape : (307511, 134)
Target Vector Shape  : (307511,)


In [14]:
# ======================================================
# Section 5: Train-Test Split
# ======================================================

from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 50)
print("Train-Test Split Completed")
print("=" * 50)

print(f"X_train Shape : {X_train.shape}")
print(f"X_test Shape  : {X_test.shape}")

print(f"y_train Shape : {y_train.shape}")
print(f"y_test Shape  : {y_test.shape}")

Train-Test Split Completed
X_train Shape : (246008, 134)
X_test Shape  : (61503, 134)
y_train Shape : (246008,)
y_test Shape  : (61503,)


In [15]:
# ======================================================
# Section 6: Calculate scale_pos_weight
# ======================================================

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()

scale_pos_weight = negative_class / positive_class

print(f"Negative Samples : {negative_class}")
print(f"Positive Samples : {positive_class}")
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

Negative Samples : 226148
Positive Samples : 19860
scale_pos_weight : 11.39


In [16]:
# ======================================================
# Section 7: Train XGBoost Model
# ======================================================

from xgboost import XGBClassifier

# Initialize the model
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,

    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    n_jobs=-1
)

# Train the model
xgb_model.fit(X_train, y_train)

print("✅ XGBoost Model Trained Successfully!")

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:NAME_CONTRACT_TYPE: str, CODE_GENDER: str, FLAG_OWN_CAR: str, FLAG_OWN_REALTY: str, NAME_TYPE_SUITE: str, NAME_INCOME_TYPE: str, NAME_EDUCATION_TYPE: str, NAME_FAMILY_STATUS: str, NAME_HOUSING_TYPE: str, OCCUPATION_TYPE: str, WEEKDAY_APPR_PROCESS_START: str, ORGANIZATION_TYPE: str, FONDKAPREMONT_MODE: str, HOUSETYPE_MODE: str, WALLSMATERIAL_MODE: str, EMERGENCYSTATE_MODE: str, AGE_GROUP: str, INCOME_GROUP: str

In [18]:
# ======================================================
# One-Hot Encode Categorical Features
# ======================================================

categorical_cols = feature_df.select_dtypes(
    include=["object", "string", "category"]
).columns

print("Categorical Columns:", len(categorical_cols))

feature_df = pd.get_dummies(
    feature_df,
    columns=categorical_cols,
    drop_first=True,
    dtype="int8"
)

print("Dataset Shape After Encoding:", feature_df.shape)

Categorical Columns: 18
Dataset Shape After Encoding: (307511, 249)


In [19]:
feature_df.dtypes.value_counts()

int8       132
float64     74
int64       43
Name: count, dtype: int64

In [20]:
feature_df.select_dtypes(include=["object", "string"]).columns

Index([], dtype='str')

In [21]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

feature_df.to_csv(
    output_dir / "application_train_feature_engineered.csv",
    index=False
)

print("✅ Updated processed dataset saved successfully!")

✅ Updated processed dataset saved successfully!


In [22]:
print("Dataset Shape:", feature_df.shape)
print("Missing Values:", feature_df.isnull().sum().sum())
print("Duplicate Rows:", feature_df.duplicated().sum())
print("String Columns:", len(feature_df.select_dtypes(include=["object", "string"]).columns))

Dataset Shape: (307511, 249)
Missing Values: 0
Duplicate Rows: 0
String Columns: 0
